# FIN-02 | Notebook 2 — Feature Engineering & Data Splits

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_all_tables
from src.features import build_feature_matrix
from src.train import prepare_splits


## 2. Load Tables

In [ ]:
data_dir = '../data'
tables = load_all_tables(data_dir)


## 3. Build Feature Matrix

In [ ]:
feature_matrix = build_feature_matrix(tables)
print("Feature Matrix Shape:", feature_matrix.shape)
print("Overall Churn Rate:", feature_matrix['churn'].mean() if 'churn' in feature_matrix else 'N/A')
display(feature_matrix.head())


## 4. Churn Label Analysis

In [ ]:
if 'churn' in feature_matrix.columns:
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    sns.countplot(data=feature_matrix, x='churn')
    plt.title('Class Distribution')
    
    plt.subplot(1, 2, 2)
    if 'tenure_months' in feature_matrix.columns:
        bands = pd.cut(feature_matrix['tenure_months'], bins=[-np.inf, 24, 48, np.inf], labels=['< 24 months', '24-48', '48+'])
        sns.barplot(x=bands, y=feature_matrix['churn'])
        plt.title('Churn Rate by Tenure Band')
    
    plt.tight_layout()
    plt.show()


## 5. Feature Distributions

In [ ]:
features_to_plot = ['trans_count', 'trans_freq_monthly', 'avg_balance', 'tenure_months']
available_features = [f for f in features_to_plot if f in feature_matrix.columns]

if available_features:
    feature_matrix[available_features].hist(bins=30, figsize=(12, 8))
    plt.suptitle('Feature Histograms')
    plt.show()

numeric_cols = feature_matrix.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(feature_matrix[numeric_cols].corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## 6. Feature vs Churn Analysis

In [ ]:
if 'churn' in feature_matrix.columns:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    if 'trans_freq_monthly' in feature_matrix.columns:
        sns.boxplot(data=feature_matrix, x='churn', y='trans_freq_monthly')
        plt.title('trans_freq_monthly by Churn')
        
    plt.subplot(1, 2, 2)
    if 'avg_balance' in feature_matrix.columns:
        sns.boxplot(data=feature_matrix, x='churn', y='avg_balance')
        plt.title('avg_balance by Churn')
        
    plt.tight_layout()
    plt.show()
    
    print("\nMean Feature Values for Churned vs Active:")
    display(feature_matrix.groupby('churn').mean(numeric_only=True))


## 7. Data Splits

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = prepare_splits(feature_matrix)

print("Split Sizes:")
print("Train:", X_train.shape[0])
print("Val:  ", X_val.shape[0])
print("Test: ", X_test.shape[0])

print("\nChurn Rates per Split:")
print(f"Train: {y_train.mean():.4f}")
print(f"Val:   {y_val.mean():.4f}")
print(f"Test:  {y_test.mean():.4f}")

print("\nWARNING: The test set is frozen. DO NOT use it for model selection or tuning!")


## 8. Save Feature Matrix

In [ ]:
os.makedirs('../data', exist_ok=True)
feature_matrix.to_parquet('../data/feature_matrix.parquet')
print("Feature matrix saved to ../data/feature_matrix.parquet")


## 9. Key Findings

- The dataset is somewhat imbalanced regarding the churn label.
- Key features like balance and transaction frequency show varying distributions for churned vs active users.
- Data splits have been prepared and test set is held out for final evaluation.